<a href="https://colab.research.google.com/github/antonum/sandbox/blob/pgvector/pgvector_hybrid_vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import psycopg2
from google.colab import userdata
#CONNECTION="postgres://tsdbadmin:xxxxxxx.yyyyy.tsdb.cloud.timescale.com:39966/tsdb?sslmode=require"
CONNECTION=userdata.get('TS_CONNECTION')
conn = psycopg2.connect(CONNECTION)
cursor = conn.cursor()

In [ ]:
import pandas as pd
def sql_results_to_df(cursor):
  columns = [desc[0] for desc in cursor.description]
  data = cursor.fetchall()
  df = pd.DataFrame(data, columns=columns)
  return df

![Screenshot 2025-02-04 at 1.33.09 PM.png](https://i.imgur.com/nYkEqMz.png)

![vbr01hE.png](https://i.imgur.com/vbr01hE.png)

In [ ]:
query = """
DROP TABLE IF EXISTS t1 CASCADE;
"""
cursor.execute(query)
conn.commit()

In [ ]:
query = """
DROP TABLE IF EXISTS t2 CASCADE;
"""
cursor.execute(query)
conn.commit()

In [ ]:
query = """
CREATE TABLE IF NOT EXISTS t1 (
  id bigserial PRIMARY KEY,
  text11 text,
  emb11 vector(3),
  text12 text,
  emb12 vector(3),
  text13 text,
  emb13 vector(3),
  filter11 int,
  filter12 char(20)
  );
"""
cursor.execute(query)
conn.commit()

query = """
CREATE TABLE IF NOT EXISTS t2 (
  id bigserial PRIMARY KEY,
  text21 text,
  emb21 vector(3),
  text22 text,
  emb22 vector(3),
  text23 text,
  emb23 vector(3),
  filter21 int,
  filter22 char(20)
  );
"""
cursor.execute(query)
conn.commit()

In [ ]:
# generate test data for t1 and t2 as insert queries
query = """
INSERT INTO t1 (text11, emb11, text12, emb12, text13, emb13, filter11, filter12) VALUES
('example text 1', '[1,2,3]', 'example text 2', '[4,5,6]', 'example text 3', '[7,8,9]', 1, 'filter_value_3'),
('another text 1', '[10,11,12]', 'another text 2', '[13,14,15]', 'another text 3', '[16,17,18]', 2, 'filter_value_3');
-- Add more rows as needed
"""
cursor.execute(query)
query = """
INSERT INTO t2 (text21, emb21, text22, emb22, text23, emb23, filter21, filter22) VALUES
('sample text 1', '[1,2,3]', 'sample text 2', '[4,5,6]', 'sample text 3', '[7,8,9]', 3, 'filter_value_3'),
('different text 1', '[10,11,12]', 'different text 2', '[13,14,15]', 'different text 3', '[16,17,18]', 4, 'filter_value_4'),
('sample text 1', '[11,21,31]', 'sample text 2', '[4,5,6]', 'sample text 3', '[7,8,9]', 3, 'filter_value_3'),
('different text 11', '[110,111,112]', 'different text 2', '[113,114,115]', 'different text 3', '[116,117,118]', 4, 'filter_value_4'),
('sample text 1', '[11,21,31]', 'sample text 2', '[4,5,6]', 'sample text 3', '[7,8,9]', 3, 'filter_value_3'),
('different text 1', '[10,11,12]', 'different text 2', '[13,14,15]', 'different text 3', '[16,17,18]', 4, 'filter_value_4'),
('sample text 1', '[12,23,34]', 'sample text 2', '[4,5,6]', 'sample text 3', '[7,8,9]', 3, 'filter_value_3'),
('different text 1', '[10,11,12]', 'different text 2', '[13,14,15]', 'different text 3', '[16,17,18]', 4, 'filter_value_4');
-- Add more rows as needed
"""
cursor.execute(query)

In [ ]:
query = """
CREATE INDEX IF NOT EXISTS emb21cosine ON t2 USING diskann (emb21 vector_cosine_ops);
"""
cursor.execute(query)

query = """
CREATE INDEX IF NOT EXISTS filter21btree ON t2 (filter21);
"""
cursor.execute(query)

conn.commit()

In [ ]:
query = """
SELECT *
FROM t2;
"""
cursor.execute(query)
sql_results_to_df(cursor)

,id,text21,emb21,text22,emb22,text23,emb23,filter21,filter22
0,1,sample text 1,"[1,2,3]",sample text 2,"[4,5,6]",sample text 3,"[7,8,9]",3,filter_value_3
1,2,different text 1,"[10,11,12]",different text 2,"[13,14,15]",different text 3,"[16,17,18]",4,filter_value_4
2,3,sample text 1,"[11,21,31]",sample text 2,"[4,5,6]",sample text 3,"[7,8,9]",3,filter_value_3
3,4,different text 11,"[110,111,112]",different text 2,"[113,114,115]",different text 3,"[116,117,118]",4,filter_value_4
4,5,sample text 1,"[11,21,31]",sample text 2,"[4,5,6]",sample text 3,"[7,8,9]",3,filter_value_3
5,6,different text 1,"[10,11,12]",different text 2,"[13,14,15]",different text 3,"[16,17,18]",4,filter_value_4
6,7,sample text 1,"[12,23,34]",sample text 2,"[4,5,6]",sample text 3,"[7,8,9]",3,filter_value_3
7,8,different text 1,"[10,11,12]",different text 2,"[13,14,15]",different text 3,"[16,17,18]",4,filter_value_4


## Simple cosine similarity

Find records in t2, most similar to specific vector in t1

In [ ]:
# find the closest matches t2.emb21<=>t1.emb11 in t2 for the rowid=1 in t1
query = """
SELECT
  t1.id as t1_id, t1.emb11, t2.id as t2_id, t2.emb21, t2.filter21,
  t2.emb21<=>t1.emb11 as distance
FROM t1, t2
WHERE
  t1.id=1
ORDER BY distance
LIMIT 10;
;
"""
cursor.execute(query)
sql_results_to_df(cursor)

,t1_id,emb11,t2_id,emb21,filter21,distance
0,1,"[1,2,3]",1,"[1,2,3]",3,0.000000
1,1,"[1,2,3]",7,"[12,23,34]",3,0.000117
2,1,"[1,2,3]",5,"[11,21,31]",3,0.000141
3,1,"[1,2,3]",3,"[11,21,31]",3,0.000141
4,1,"[1,2,3]",6,"[10,11,12]",4,0.048742
5,1,"[1,2,3]",2,"[10,11,12]",4,0.048742
6,1,"[1,2,3]",8,"[10,11,12]",4,0.048742
7,1,"[1,2,3]",4,"[110,111,112]",4,0.071425


In [ ]:
# just for debugging
conn.commit()

## Multiple distances, additional filter

Retrieve two distances, filter by non-vector field

In [ ]:
# calculate two metrics. Note that cut off (LIMIT 10) will work only for one.
query = """
SELECT
  t1.id as t1_id, t1.emb11, t2.id as t2_id, t2.emb21, t2.filter21,
  t2.emb21<=>t1.emb11 as distance1,
  t2.emb22<=>t1.emb12 as distance2
FROM t1, t2
WHERE
  t1.id=1
  AND t2.filter21=4 -- filter applied
ORDER BY distance1
LIMIT 10;

"""
cursor.execute(query)
sql_results_to_df(cursor)

,t1_id,emb11,t2_id,emb21,filter21,distance1,distance2
0,1,"[1,2,3]",2,"[10,11,12]",4,0.048742,0.005363
1,1,"[1,2,3]",6,"[10,11,12]",4,0.048742,0.005363
2,1,"[1,2,3]",8,"[10,11,12]",4,0.048742,0.005363
3,1,"[1,2,3]",4,"[110,111,112]",4,0.071425,0.011943


## Final query

Subquery (from example above) selects 50 potential matches with all the distances calculated and filters applied. 50 results cut-off is based on the distance1 as it has a higher weight.

Outer query adds weights to the distances and performs final reranking, based on the similarity score, returning 10 best matches.

In [ ]:
# Add weights to two distance metrics. consolidate in similarity score
query = """
SELECT (1-distance1*3)*(1-distance2*0.5) as similarity, -- arbitrary weights added to the distance metrics
  t1_id, t2_id, distance1, distance2
FROM (
  SELECT
    t1.id as t1_id, t1.emb11, t2.id as t2_id, t2.emb21, t2.filter21,
    t2.emb21<=>t1.emb11 as distance1,
    t2.emb22<=>t1.emb12 as distance2
  FROM t1, t2
  WHERE
    t1.id=1
    AND t2.filter21=4 --arbitrary filter here
  ORDER BY distance1
  LIMIT 50 -- select higher then needed number of potential matches
)
ORDER BY similarity DESC -- reranking, based on similarity score
LIMIT 10; -- actual matches based on similarity score
"""
cursor.execute(query)
sql_results_to_df(cursor)

,similarity,t1_id,t2_id,distance1,distance2
0,0.851485,1,2,0.048742,0.005363
1,0.851485,1,6,0.048742,0.005363
2,0.851485,1,8,0.048742,0.005363
3,0.781034,1,4,0.071425,0.011943


## Vector index with Streaming DiskANN

This notebook is using [pgvectorscale](https://github.com/timescale/pgvectorscale) extension for vector indexing. `USING diskann` in `CREATE INDEX` statement.

```
CREATE INDEX IF NOT EXISTS emb21cosine ON t2 USING diskann (emb21 vector_cosine_ops);
```

The default settings of Streaming DiskANN already works with filtering on non-vector fields and LIMIT as expected. Streaming nature of index would make sure that no matter how selective the filter is - you'll get up to the LIMIT matches from the query. No additional parameter/setting tuning is nessesary. HNSW index on the other hand would require additional tuning to accomodate filer+vector combination (see below).

Additionally, for this specific case, since the outer query is already doing reranking, you can further optimize the inner query by setting the low `diskann.query_search_list_size` and `diskann.query_rescore=0` to minimize/ skip the reranking step, performed by DiskANN index.

## Indexing/limit consideration for PGVector HNSW

https://github.com/pgvector/pgvector?tab=readme-ov-file#filtering

https://github.com/pgvector/pgvector?tab=readme-ov-file#iterative-index-scans

With approximate indexes, filtering is applied after the index is scanned. If a condition matches 10% of rows, with HNSW and the default hnsw.ef_search of 40, only 4 rows will match on average. For more rows, increase hnsw.ef_search.
```
SET hnsw.ef_search = 200;
```

With approximate indexes, queries with filtering can return less results since filtering is applied after the index is scanned. Starting with 0.8.0, you can enable iterative index scans, which will automatically scan more of the index until enough results are found (or it reaches hnsw.max_scan_tuples or ivfflat.max_probes).

Iterative scans can use strict or relaxed ordering.

Strict ensures results are in the exact order by distance
```
SET hnsw.iterative_scan = strict_order;
```
Relaxed allows results to be slightly out of order by distance, but provides better recall
```
SET hnsw.iterative_scan = relaxed_order;
```